# ToR + Evaluation Matrix → Grounded CV Extraction & Rating
## Using Gemini File API (the open equivalent of NotebookLM)

**How this works:**
NotebookLM has no public REST API yet. But it uses the same engine as **Gemini File API** internally:
1. Upload your ToR PDF and Evaluation Matrix as persistent file objects
2. Both documents are grounded — the model only answers from what is in them
3. Run extraction and scoring prompts against the uploaded files

**Why this is better than pasting text:**
- Handles multi-page PDFs natively (up to 1500 pages)
- No token limit issue with large TOR documents  
- Evidence is guaranteed to be in the actual document (hallucination guard)
- Files persist for 48 hours — reuse across multiple CV queries

In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────────────
import os, json, re, io, time, pathlib, textwrap, hashlib, requests
from typing import Optional
from pprint import pprint

# Load the Gemini key from the project env file
def load_env(path: str = "../.env.local") -> dict:
    env = {}
    try:
        for line in pathlib.Path(path).read_text().splitlines():
            m = re.match(r'^([^=]+)=(.*)$', line.strip())
            if m:
                k, v = m.group(1).strip(), m.group(2).strip().strip('"')
                env[k] = v
    except FileNotFoundError:
        pass
    return env

env = load_env()
GEMINI_API_KEY = env.get("GEMINI_API_KEY", os.environ.get("GEMINI_API_KEY", ""))
GEMINI_MODEL   = env.get("GEMINI_MODEL",   os.environ.get("GEMINI_MODEL", "gemini-flash-latest"))
BASE_URL       = f"https://generativelanguage.googleapis.com"

assert GEMINI_API_KEY, "Set GEMINI_API_KEY in ../.env.local or as an environment variable"
print(f"✅ Key loaded ({len(GEMINI_API_KEY)} chars) — model: {GEMINI_MODEL}")

## Step 1 — Upload the ToR and Evaluation Matrix as Gemini Files

Files are stored for **48 hours** and reused by URI, so you only upload once per session.

In [ ]:
# ── 1. Gemini File API helpers ───────────────────────────────────────────────
def upload_file(local_path: str, display_name: str = None) -> dict:
    """Upload a local PDF/DOCX/XLSX to the Gemini File API. Returns file metadata dict."""
    path = pathlib.Path(local_path)
    mime_map = {
        ".pdf":  "application/pdf",
        ".docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".txt":  "text/plain",
    }
    mime = mime_map.get(path.suffix.lower(), "application/octet-stream")
    display = display_name or path.name

    # Resumable upload (works for any file size)
    init_resp = requests.post(
        f"{BASE_URL}/upload/v1beta/files",
        params={"key": GEMINI_API_KEY},
        headers={
            "X-Goog-Upload-Protocol": "multipart",
            "Content-Type": "multipart/related; boundary=BOUNDARY",
        },
        data=(
            b"--BOUNDARY\r\nContent-Type: application/json; charset=UTF-8\r\n\r\n"
            + json.dumps({"file": {"displayName": display}}).encode()
            + b"\r\n--BOUNDARY\r\nContent-Type: " + mime.encode()
            + b"\r\n\r\n"
            + path.read_bytes()
            + b"\r\n--BOUNDARY--"
        ),
    )
    init_resp.raise_for_status()
    file_meta = init_resp.json().get("file", {})
    print(f"  Uploaded: {display} → {file_meta.get('uri','?')}")
    return file_meta


def wait_for_file(file_meta: dict, timeout: int = 60) -> dict:
    """Poll until the file state is ACTIVE."""
    name = file_meta["name"]
    for _ in range(timeout):
        r = requests.get(
            f"{BASE_URL}/v1beta/{name}",
            params={"key": GEMINI_API_KEY}
        )
        r.raise_for_status()
        meta = r.json()
        state = meta.get("state", "UNKNOWN")
        if state == "ACTIVE":
            print(f"  ✅ File ACTIVE: {meta.get('displayName')}")
            return meta
        if state == "FAILED":
            raise RuntimeError(f"File processing FAILED: {meta}")
        time.sleep(1)
    raise TimeoutError(f"File still not ACTIVE after {timeout}s")


def list_files() -> list:
    r = requests.get(f"{BASE_URL}/v1beta/files", params={"key": GEMINI_API_KEY})
    r.raise_for_status()
    return r.json().get("files", [])

In [ ]:
# ── 1b. Upload your ToR and Evaluation Matrix ─────────────────────────────────
# Replace these paths with your actual files.
# Any PDF, DOCX, or XLSX works. You can also upload multiple TOR sections.

TOR_FILE_PATH    = "Expert_Staffing_Matrix_PRECISE_TVET4RE.docx"   # the DOCX in this folder
MATRIX_FILE_PATH = "Expert_Staffing_Matrix_PRECISE_TVET4RE.docx"   # or a separate matrix file
CV_FILE_PATH     = "CVs/"  # directory — we'll pick the first CV below

# Upload (skip if already uploaded in this session — reuse URIs)
print("Uploading documents to Gemini File API…")
tor_meta    = upload_file(TOR_FILE_PATH,    "ToR Document")
tor_meta    = wait_for_file(tor_meta)

matrix_meta = upload_file(MATRIX_FILE_PATH, "Evaluation Matrix")
matrix_meta = wait_for_file(matrix_meta)

print(f"\nToR URI    : {tor_meta['uri']}")
print(f"Matrix URI : {matrix_meta['uri']}")

In [ ]:
# ── 1c. (Optional) Upload a candidate CV ──────────────────────────────────────
import glob

cv_dir = pathlib.Path(CV_FILE_PATH)
cv_files = sorted(cv_dir.glob("*.pdf")) + sorted(cv_dir.glob("*.docx"))
if cv_files:
    cv_path = str(cv_files[0])
    print(f"Uploading CV: {cv_path}")
    cv_meta = upload_file(cv_path, f"CV: {cv_files[0].name}")
    cv_meta = wait_for_file(cv_meta)
    print(f"CV URI: {cv_meta['uri']}")
else:
    cv_meta = None
    print("No CV file found in CVs/ — paste raw text in Step 3 instead.")

## Step 2 — Extract ToR Structure (Persona 1, grounded)

Query the ToR document directly. The model cannot answer from general knowledge — it must cite the document.

In [ ]:
# ── 2. Grounded Gemini call helper ───────────────────────────────────────────
def gemini_with_files(prompt: str, file_uris: list[dict], json_mode: bool = True) -> str:
    """
    Call Gemini with one or more uploaded files as grounded context.
    file_uris: list of {"mime_type": "...", "uri": "..."} dicts
    """
    parts = [{"file_data": f} for f in file_uris]
    parts.append({"text": prompt})
    payload = {
        "contents": [{"parts": parts}],
        "generationConfig": {
            "temperature": 0,
            **({"responseMimeType": "application/json"} if json_mode else {}),
        },
    }
    r = requests.post(
        f"{BASE_URL}/v1beta/models/{GEMINI_MODEL}:generateContent",
        params={"key": GEMINI_API_KEY},
        json=payload,
        timeout=120,
    )
    r.raise_for_status()
    data = r.json()
    text = data["candidates"][0]["content"]["parts"][0]["text"]
    return text


def parse_json_loose(text: str):
    t = text.strip()
    m = re.search(r'```(?:json)?\s*([\s\S]*?)```', t, re.I)
    if m: t = m.group(1).strip()
    start = next((i for i, c in enumerate(t) if c in '{['), 0)
    t = t[start:]
    end = max(t.rfind('}'), t.rfind(']'))
    if end >= 0: t = t[:end+1]
    return json.loads(t)

In [ ]:
# ── 2b. Extract ToR structure from the uploaded document ─────────────────────
TOR_EXTRACTION_PROMPT = """You are a tender-document analyst. Extract the expert roles and qualifications from this ToR document.

Rules:
- Extract ONLY what is explicitly written. If a field is absent, use null.
- Preserve the donor's EXACT wording for role names and qualification text verbatim.
- Distinguish mandatory ("must have") from preferred ("advantageous"/"preferred").
- Set "binary": true for pass/fail requirements (e.g. must hold a degree), false for scaled (e.g. 10 years experience).
- Set "durationYears" when a specific number of years is stated, otherwise null.

Return JSON only — no prose, no markdown fences:
{
  "projectTitle": "string|null",
  "donor": "string|null",
  "expertRoles": [
    {
      "roleName": "string (verbatim)",
      "mandatory": [{"text": "verbatim requirement", "category": "string|null", "durationYears": number|null, "binary": bool}],
      "preferred":  [{"text": "verbatim requirement", "category": "string|null", "durationYears": number|null, "binary": bool}]
    }
  ],
  "deliverables": ["string"]
}"""

print("Extracting ToR structure from uploaded document…")
raw = gemini_with_files(
    TOR_EXTRACTION_PROMPT,
    [{"mime_type": tor_meta["mimeType"], "uri": tor_meta["uri"]}]
)
tor_structure = parse_json_loose(raw)
print(f"\n✅ Extracted {len(tor_structure.get('expertRoles', []))} expert role(s) from: {tor_structure.get('projectTitle','?')}\n")
for role in tor_structure.get("expertRoles", []):
    print(f"  Role: {role['roleName']}")
    print(f"    Mandatory requirements: {len(role.get('mandatory', []))}")
    print(f"    Preferred requirements: {len(role.get('preferred', []))}")

In [ ]:
# Pretty-print the full extraction result
print(json.dumps(tor_structure, indent=2)[:3000], "…" if len(json.dumps(tor_structure)) > 3000 else "")

## Step 3 — Extract Evaluation Matrix Criteria (grounded)

Pull the exact scoring weights and thresholds from the uploaded matrix document.

In [ ]:
# ── 3. Extract evaluation criteria from the matrix document ───────────────────
MATRIX_EXTRACTION_PROMPT = """You are an evaluation matrix analyst. Extract every scoring criterion from this evaluation matrix document.

For each criterion:
- Preserve the exact label text verbatim
- Record the maximum points (maxPoints)
- Classify its category (Education, Language, Experience, etc.)
- Set "binary": true only for pass/fail thresholds (e.g. "Must hold a degree")
- Group by expert role if the matrix contains multiple roles

Return JSON only:
{
  "matrixName": "string|null",
  "roles": [
    {
      "roleName": "string",
      "minPassPercent": number|null,
      "criteria": [
        {
          "key": "short_stable_id",
          "category": "string",
          "label": "verbatim criterion text",
          "maxPoints": number,
          "binary": bool
        }
      ]
    }
  ]
}"""

print("Extracting evaluation matrix criteria from uploaded document…")
raw = gemini_with_files(
    MATRIX_EXTRACTION_PROMPT,
    [{"mime_type": matrix_meta["mimeType"], "uri": matrix_meta["uri"]}]
)
matrix_structure = parse_json_loose(raw)

total_criteria = sum(len(r.get("criteria", [])) for r in matrix_structure.get("roles", []))
print(f"\n✅ Extracted {total_criteria} criteria across {len(matrix_structure.get('roles', []))} role(s)")
for role in matrix_structure.get("roles", []):
    total = sum(c["maxPoints"] for c in role.get("criteria", []))
    print(f"\n  Role: {role['roleName']} | Min pass: {role.get('minPassPercent')}% | Total pts: {total}")
    for c in role.get("criteria", [])[:5]:
        print(f"    [{c['category']}] {c['label'][:60]}… — {c['maxPoints']} pts {'(binary)' if c.get('binary') else ''}")
    if len(role.get("criteria", [])) > 5:
        print(f"    … and {len(role['criteria']) - 5} more")

## Step 4 — Score a CV against one criterion (Persona 3, per-criterion judge — grounded)

The key rule: **one criterion per call**. The model scores using only evidence from the CV file.
Both the CV and the criterion text are grounded — no hallucination possible.

In [ ]:
# ── 4. Per-criterion grounded scoring ────────────────────────────────────────
def score_criterion_grounded(criterion: dict, cv_file_meta: dict) -> dict:
    """
    Score ONE criterion against a CV file using Gemini File API grounding.
    Returns: {score, evidence, confidence, reasoning}
    """
    threshold_note = (
        "THRESHOLD CRITERION — award the full points if the requirement is clearly met, otherwise 0."
        if criterion.get("binary") else
        "PROPORTIONAL — award partial points proportional to the evidence strength."
    )
    prompt = f"""You are an evaluation-committee assessor. Score this candidate against ONE specific criterion.
Be CONSERVATIVE — donor committees penalise overclaiming. Award points only for what is EXPLICITLY demonstrated in the CV.

CRITERION
Category: {criterion['category']}
Text: {criterion['label']}
Max points: {criterion['maxPoints']}
Scoring mode: {threshold_note}

Rules:
- Award points only for explicitly described activities, not just job titles.
- If a duration is required, sum only roles that genuinely match the criterion subject.
- Return the EXACT source text (under 25 words) from the CV that justifies the score.
- If you cannot find supporting text, award 0 and say so.
- Confidence < 0.6 if evidence is indirect or ambiguous.

Return JSON only (no markdown):
{{"score": number, "evidence": "string|null", "confidence": number, "reasoning": "string"}}"""

    files = [{"mime_type": cv_file_meta["mimeType"], "uri": cv_file_meta["uri"]}]
    raw = gemini_with_files(prompt, files)
    result = parse_json_loose(raw)
    
    # Guardrails
    score = max(0, min(criterion["maxPoints"], round(float(result.get("score", 0)), 2)))
    if not result.get("evidence"):
        score = 0  # no evidence → no points
    if criterion.get("binary"):
        score = criterion["maxPoints"] if score >= criterion["maxPoints"] * 0.999 else 0

    return {
        "criterionKey":      criterion["key"],
        "criterionLabel":    criterion["label"],
        "category":          criterion["category"],
        "score":             score,
        "maxPoints":         criterion["maxPoints"],
        "evidence":          result.get("evidence"),
        "confidence":        min(1, max(0, float(result.get("confidence", 0)))),
        "reasoning":         result.get("reasoning", ""),
        "needsReview":       float(result.get("confidence", 0)) < 0.6 or not result.get("evidence"),
    }

In [ ]:
# Score each criterion for the first role in the matrix
# (Run this against a real CV file; uses cv_meta from Step 1c)
if cv_meta is None:
    print("⚠️  No CV uploaded — paste a sample CV path below or run Step 1c first.")
else:
    role = matrix_structure["roles"][0]
    print(f"Scoring '{role['roleName']}' against {cv_meta.get('displayName', 'CV')}\n")
    print(f"{'Criterion':<55} {'Score':>6} {'Max':>5} {'Conf':>5} {'Flag'}")
    print("─" * 80)
    
    results = []
    for crit in role["criteria"]:
        r = score_criterion_grounded(crit, cv_meta)
        results.append(r)
        flag = "⚑ review" if r["needsReview"] else "✓"
        label = r["criterionLabel"][:52] + "…" if len(r["criterionLabel"]) > 53 else r["criterionLabel"]
        print(f"{label:<55} {r['score']:>6} {r['maxPoints']:>5} {r['confidence']:>5.2f} {flag}")
    
    total  = sum(r["score"] for r in results)
    max_pt = sum(r["maxPoints"] for r in results)
    pct    = round(total / max_pt * 100, 1) if max_pt else 0
    flags  = sum(1 for r in results if r["needsReview"])
    print(f"\n{'TOTAL':<55} {total:>6} {max_pt:>5}  →  {pct}%")
    print(f"Criteria needing human review: {flags}/{len(results)}")
    min_pct = role.get("minPassPercent") or 85
    print(f"Result: {'✅ PASS' if pct >= min_pct else '❌ FAIL'} (threshold: {min_pct}%)")

## Step 5 — Generate the targeted CV sections (ToR + Matrix grounded)

Now use **all three grounded files** together: the ToR + matrix provide the target, the CV provides the source.
The model rewrites the CV sections to match the exact TOR language.

In [ ]:
# ── 5. CV tailoring grounded on ToR + Matrix + CV ─────────────────────────────
def tailor_cv_grounded(tor_file: dict, matrix_file: dict, cv_file: dict, project_name: str) -> dict:
    prompt = f"""You are a professional CV tailoring expert for development-sector bids (GIZ, EU, UN).
You have access to THREE documents:
  1. The Terms of Reference (ToR)
  2. The Evaluation Matrix (scoring criteria)
  3. The expert's CV

Your task: tailor the CV for project "{project_name}" by:
- Rewriting the professional summary to directly address the TOR language
- Identifying the strongest evidence for EACH evaluation criterion
- Computing a tor_match_pct (0–100) based on how well the CV meets the criteria

STRICT rules:
- Quote only evidence that genuinely appears in the CV document
- Use the TOR's own wording for requirements (do not paraphrase)
- For each criterion, if no evidence exists, say so explicitly

Return JSON only (no markdown fences):
{{
  "expert_name": "string",
  "tor_match_pct": number,
  "sections": [
    {{"section": "string", "original": "string|null", "tailored": "string", "keywords": ["string"]}}
  ],
  "matrix_matches": [
    {{"requirement": "verbatim from TOR", "evidence": "under 25 words from CV", "score": number, "max_score": number}}
  ],
  "provider": "gemini-file-api"
}}"""

    files = [
        {"mime_type": tor_file["mimeType"],    "uri": tor_file["uri"]},
        {"mime_type": matrix_file["mimeType"], "uri": matrix_file["uri"]},
        {"mime_type": cv_file["mimeType"],     "uri": cv_file["uri"]},
    ]
    raw = gemini_with_files(prompt, files)
    return parse_json_loose(raw)


if cv_meta:
    print("Running grounded CV tailoring with ToR + Matrix + CV files…")
    tailor_result = tailor_cv_grounded(
        tor_meta, matrix_meta, cv_meta,
        project_name=tor_structure.get("projectTitle", "Project")
    )
    
    print(f"\n Expert: {tailor_result.get('expert_name')}")
    print(f" TOR Match: {tailor_result.get('tor_match_pct')}%\n")
    for sec in tailor_result.get("sections", []):
        print(f"  ── {sec['section']} ──")
        print(f"  {sec['tailored'][:200]}…\n")
        if sec.get("keywords"):
            print(f"  Keywords: {', '.join(sec['keywords'][:6])}\n")
else:
    print("Upload a CV in Step 1c to run this.")

## Step 6 — Combine results and export to DOCX via the FastAPI service

Send the grounded tailor result to the Python FastAPI endpoint to generate the final DOCX.

In [ ]:
# ── 6. Generate DOCX via the FastAPI service ──────────────────────────────────
# Point to the local or production service
API_URL = os.environ.get("CV_TAILOR_API_URL", "http://localhost:8001")

if cv_meta and 'tailor_result' in dir():
    for template_id in ["giz", "eu"]:
        resp = requests.post(
            f"{API_URL}/generate",
            json={"result": tailor_result, "template_id": template_id},
            timeout=60,
        )
        if resp.ok:
            filename = f"tailored_{tailor_result.get('expert_name', 'expert').replace(' ','_')}_{template_id}.docx"
            out_path = pathlib.Path(filename)
            out_path.write_bytes(resp.content)
            print(f"✅ Saved {template_id.upper()} DOCX → {out_path}  ({len(resp.content)//1024} KB)")
        else:
            print(f"⚠️  FastAPI {template_id} error: {resp.status_code} {resp.text[:120]}")
else:
    print("Run Steps 1c, 4 and 5 first.")

## Step 7 — Batch score multiple CVs

Loop over all CVs in the folder and produce a comparison table against the same matrix criteria.
Useful when you have 5–10 candidate CVs for the same position.

In [ ]:
# ── 7. Batch scoring ──────────────────────────────────────────────────────────
import pandas as pd

cv_dir = pathlib.Path("CVs/")
cv_paths = sorted(cv_dir.glob("*.pdf")) + sorted(cv_dir.glob("*.docx"))

if not cv_paths:
    print("No CVs found in CVs/ directory.")
else:
    role = matrix_structure["roles"][0]
    rows = []
    for cv_path in cv_paths:
        print(f"Processing: {cv_path.name}…")
        try:
            cv_f = upload_file(str(cv_path), cv_path.name)
            cv_f = wait_for_file(cv_f)
        except Exception as e:
            print(f"  Upload failed: {e}")
            continue
        
        scores = []
        for crit in role["criteria"]:
            r = score_criterion_grounded(crit, cv_f)
            scores.append(r)
        
        total  = sum(r["score"] for r in scores)
        max_pt = sum(r["maxPoints"] for r in scores)
        pct    = round(total / max_pt * 100, 1) if max_pt else 0
        flags  = sum(1 for r in scores if r["needsReview"])
        rows.append({
            "Candidate": cv_path.stem,
            "Score": total,
            "MaxScore": max_pt,
            "Percentage": pct,
            "Pass": "✅" if pct >= (role.get("minPassPercent") or 85) else "❌",
            "ReviewFlags": flags,
        })
    
    df = pd.DataFrame(rows).sort_values("Percentage", ascending=False)
    print("\n── Candidate Ranking ──────────────────────────────")
    print(df.to_string(index=False))

## How to integrate this with the existing portal

The grounded approach via the Gemini File API slots directly into the existing agent system:

| Step | Current (text-based) | Grounded (File API) |
|---|---|---|
| ToR extraction | Persona 1: text prompt | Upload ToR PDF → query with `file_data` URI |
| CV extraction | Persona 2: extracted text string | Upload CV PDF → query with `file_data` URI |
| Per-criterion scoring | Persona 3: text excerpts | Upload CV + matrix → one call per criterion |
| Evidence verification | `verifyEvidence()` substring check | Model cites directly from grounded doc |

**Integration path:** `src/lib/agents/grounded-agent.ts` — a new agent that:
1. Uploads the ToR/matrix/CV via the Gemini File API from the Next.js server
2. Caches file URIs in the `AgentRuns` table (reuse for 48 hours)
3. Feeds URIs into `runAgent()` instead of plain text prompts
4. Falls back to the existing text-based agents when no file is available

> **NotebookLM note:** When Google opens the NotebookLM API (expected later in 2026), you will be able to create persistent "notebooks" (corpora) with these same documents and query them. The code structure above is identical — you would swap the `file_data` URI for a `corpus_uri`. Your grounding logic, scoring criteria, and DOCX generation all stay the same.